# Coherence Change Detection for Earthquake Damage Mapping
### Case study: 2016 Amatrice–Norcia earthquake sequence (Central Italy)

This notebook replaces the LOS-displacement version of use case 1 with the **coherence
change-detection** approach: instead of converting phase to displacement, it compares
**pre-event vs. post-event interferometric coherence** to flag abrupt surface disruption —
collapsed buildings, rubble, landslides triggered by shaking — which is the more direct and
much cheaper rapid-response product after a damaging earthquake.

Built entirely from openEO User-Defined Processes (UDPs) on the Copernicus Data Space
Ecosystem (CDSE):

- [`sentinel1_sar_coherence`](https://algorithm-catalogue.apex.esa.int/apps/sentinel1_sar_coherence) — geocoded coherence, used twice: once for a "quiet" pre-event baseline pair, once for a pair straddling the event
- `SENTINEL1_GRD` + `sar_backscatter` — pre/post amplitude, used as a cross-check

**Why coherence instead of displacement here?** Collapsed structures and rubble fields
decorrelate strongly and unpredictably (they are no longer the same physical scatterer from
one acquisition to the next), which coherence picks up directly and robustly. LOS
displacement, by contrast, is only meaningful where the ground/structure surface is *still
coherent enough to unwrap* — exactly the pixels that are least informative about collapse.
Coherence change detection is also purely pairwise, with no phase unwrapping at all, so it is
even cheaper and faster to compute than the displacement-snapshot approach.

**Area of interest:** Amatrice, Central Italy — epicentral area of the Mw 6.2 earthquake of
24 August 2016.

**What this notebook does:**
1. Discovers a Sentinel-1 burst covering the AOI.
2. Lists available acquisition dates so you can pick a real baseline pair (fully pre-event)
   and a real event pair (straddling 24 Aug 2016).
3. Runs `sentinel1_sar_coherence` for both pairs.
4. Loads pre/post-event `SENTINEL1_GRD` amplitude as a cross-check signal.
5. Applies a **custom UDF** that computes the coherence drop (Δγ), combines it with the
   amplitude change, and classifies pixels into damage-severity classes.
6. Visualises the resulting damage-severity map.

> As before, this is a template — the burst_id and the exact acquisition dates need to be
> confirmed from the catalogue (steps 2–3) before running the job.


In [ ]:
# --- Imports ---
import requests
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import openeo
import openeo.processes


In [ ]:
# --- Connect to the openEO back-end on CDSE ---
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()


## 1. Area of interest

A small box around Amatrice, Italy (epicentre of the 24 Aug 2016 Mw 6.2 event).

In [ ]:
aoi = {
    "type": "Polygon",
    "coordinates": [
        [
            [13.10, 42.60],
            [13.10, 42.78],
            [13.38, 42.78],
            [13.38, 42.60],
            [13.10, 42.60],
        ]
    ],
}


## 2. Find a Sentinel-1 burst covering the AOI

`sentinel1_sar_coherence` operates on individual Sentinel-1 TOPS bursts, so we need a
`burst_id` and `sub_swath` for our AOI.

Docs: https://documentation.dataspace.copernicus.eu/APIs/Sentinel-1%20SLC%20Burst.html


In [ ]:
def find_candidate_bursts(aoi_polygon, start, end, top=20):
    coords = aoi_polygon["coordinates"][0]
    wkt_coords = ", ".join(f"{lon} {lat}" for lon, lat in coords)
    footprint_wkt = f"POLYGON(({wkt_coords}))"

    filter_str = (
        f"OData.CSC.Intersects(area=geography'SRID=4326;{footprint_wkt}') "
        f"and ContentDate/Start gt {start}T00:00:00.000Z "
        f"and ContentDate/Start lt {end}T00:00:00.000Z "
        f"and PolarisationChannels eq 'VV'"
    )
    url = (
        "https://catalogue.dataspace.copernicus.eu/odata/v1/Bursts"
        f"?$filter={filter_str}&$top={top}&$orderby=ContentDate/Start asc"
    )
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.json().get("value", [])


# Search a generous window around the mainshock so we have pre- and post-event candidates
candidates = find_candidate_bursts(aoi, "2016-07-15", "2016-09-15")

for b in candidates:
    print(
        b.get("Id"), "|",
        "burst_id:", b.get("BurstId"),
        "swath:", b.get("SwathIdentifier"),
        "orbit:", b.get("RelativeOrbitNumber"),
        "direction:", b.get("OrbitDirection"),
        "date:", b.get("ContentDate", {}).get("Start"),
    )


> **Note:** as before, the Sentinel-1 SLC Burst catalogue reliably indexes bursts via this
> OData endpoint from August 2024 onward; for a 2016 event, cross-check exact acquisition
> dates/orbit via the general `SENTINEL-1` catalogue (`Collection eq 'SENTINEL-1'`,
> `ProductType eq 'IW_SLC__1S'`) and supply `burst_id`/`sub_swath` manually below.

## 3. Fix the processing parameters

- `BASELINE_PAIR` — two acquisitions, **both before** the earthquake, giving a "quiet"
  reference coherence level for this terrain/track.
- `EVENT_PAIR` — two acquisitions **straddling** the mainshock (24 Aug 2016).

Sentinel-1 was on its 12-day repeat cycle over this area in August 2016 (S1B not yet fully
operational), so both pairs should be spaced ~12 days apart on the same track.


In [ ]:
BURST_ID = 123456          # <-- replace with a real burst_id for the AOI
SUB_SWATH = "IW2"          # <-- replace as needed

# both dates pre-event -> baseline / "quiet" coherence
BASELINE_PAIR = ["2016-07-13", "2016-07-25"]
# straddles the 24 Aug mainshock
EVENT_PAIR = ["2016-08-12", "2016-08-24"]

COHERENCE_WINDOW_AZ = 2
COHERENCE_WINDOW_RG = 10


## 4. Run `sentinel1_sar_coherence` for both pairs

Two independent, single-pair coherence products — no time-series inversion.

In [ ]:
def load_coherence_pair(pair_dates, label):
    cube = connection.datacube_from_process(
        "sentinel1_sar_coherence",
        namespace=(
            "https://raw.githubusercontent.com/ESA-APEx/apex_algorithms/refs/heads/main/"
            "algorithm_catalog/eurac/sentinel1_sar_coherence/openeo_udp/"
            "sentinel1_sar_coherence.json"
        ),
        **{
            "temporal_extent": pair_dates,
            "temporal_baseline": 12,
            "burst_id": BURST_ID,
            "coherence_window_az": COHERENCE_WINDOW_AZ,
            "coherence_window_rg": COHERENCE_WINDOW_RG,
            "polarization": "VV",
            "sub_swath": SUB_SWATH,
        },
    )
    return cube.rename_labels(dimension="bands", target=[label])


coherence_baseline = load_coherence_pair(BASELINE_PAIR, "coherence_baseline")
coherence_event = load_coherence_pair(EVENT_PAIR, "coherence_event")

coherence_pairs = coherence_baseline.merge_cubes(coherence_event)


## 5. Load pre/post-event amplitude as a cross-check

A drop in coherence caused by real ground/structure disruption is usually accompanied by a
backscatter change too (rubble, collapsed roofs, or debris have different scattering
properties than intact structures). This is the same `sar_backscatter` pattern as the
CDSE oil-spill mapping example.

In [ ]:
def load_amplitude(date, label, window_days=2):
    from datetime import date as _date, timedelta

    d = _date.fromisoformat(date)
    start = (d - timedelta(days=window_days)).isoformat()
    end = (d + timedelta(days=window_days)).isoformat()

    cube = connection.load_collection(
        "SENTINEL1_GRD",
        temporal_extent=[start, end],
        spatial_extent=aoi,
        bands=["VV"],
    )
    cube = cube.sar_backscatter(coefficient="sigma0-ellipsoid")
    cube = cube.apply(process=lambda data: 10 * openeo.processes.log(data, base=10))
    cube = cube.reduce_dimension(dimension="t", reducer="mean")
    return cube.rename_labels(dimension="bands", target=[label])


amplitude_pre = load_amplitude(BASELINE_PAIR[1], "amplitude_pre")
amplitude_post = load_amplitude(EVENT_PAIR[1], "amplitude_post")

amplitude_pair = amplitude_pre.merge_cubes(amplitude_post)


The coherence products (burst-level SAR geometry, geocoded) and the GRD amplitude
products (regular grid) are generated by different processing chains, so their exact pixel
grids may not match. Resample the amplitude cube onto the coherence cube's grid before
merging:

In [ ]:
amplitude_pair_resampled = amplitude_pair.resample_cube_spatial(coherence_pairs)

damage_inputs = coherence_pairs.merge_cubes(amplitude_pair_resampled)


## 6. UDF: coherence-drop + amplitude-change damage classification

For each pixel:
- `delta_coherence = coherence_baseline - coherence_event` — how much *more* decorrelated
  the pixel became across the event, relative to its own pre-event stability,
- `delta_amplitude = amplitude_post - amplitude_pre` (dB) — backscatter change,
- combine both into a 3-class severity label using simple, transparent thresholds
  (tunable per region/terrain):
  - **0 — no significant change**
  - **1 — partial disruption** (moderate coherence drop, and/or amplitude change)
  - **2 — severe disruption** (large coherence drop *and* a large amplitude change —
    higher-confidence damage signature)


In [ ]:
damage_classification_udf = """
import numpy as np
import xarray as xr

DELTA_COH_PARTIAL = 0.15   # moderate coherence drop
DELTA_COH_SEVERE = 0.35    # large coherence drop
DELTA_AMP_THRESHOLD = 2.0  # dB, amplitude change considered significant


def apply_datacube(cube: xr.DataArray, context: dict) -> xr.DataArray:
    coh_baseline = cube.sel(bands="coherence_baseline")
    coh_event = cube.sel(bands="coherence_event")
    amp_pre = cube.sel(bands="amplitude_pre")
    amp_post = cube.sel(bands="amplitude_post")

    delta_coherence = coh_baseline - coh_event
    delta_amplitude = np.abs(amp_post - amp_pre)

    severity = xr.zeros_like(delta_coherence)

    partial = (delta_coherence >= DELTA_COH_PARTIAL) | (delta_amplitude >= DELTA_AMP_THRESHOLD)
    severe = (delta_coherence >= DELTA_COH_SEVERE) & (delta_amplitude >= DELTA_AMP_THRESHOLD)

    severity = xr.where(partial, 1, severity)
    severity = xr.where(severe, 2, severity)

    result = xr.concat([severity, delta_coherence, delta_amplitude], dim="bands")
    result = result.assign_coords(
        bands=["damage_severity", "delta_coherence", "delta_amplitude_db"]
    )
    return result
"""


In [ ]:
s1_damage_map = damage_inputs.apply_dimension(
    code=damage_classification_udf,
    runtime="Python",
    dimension="bands",
)


## 7. Execute the batch job

In [ ]:
job = s1_damage_map.create_job(
    title="amatrice_damage_mapping",
    outputformat="netCDF",
)
job.start_and_wait()
job.get_results().download_files("amatrice_damage")


## 8. Plot the damage-severity map

In [ ]:
result_ds = xr.load_dataset("amatrice_damage/openEO.nc")

severity = result_ds["damage_severity"] if "damage_severity" in result_ds else (
    result_ds.to_array(dim="bands").sel(bands="damage_severity")
)

labels = ["No significant change", "Partial disruption", "Severe disruption"]
colors = ["#2b2b2b", "#fdae61", "#d7191c"]
cmap = plt.matplotlib.colors.ListedColormap(colors)

fig, ax = plt.subplots(figsize=(6, 6), dpi=100)
severity.squeeze().plot.imshow(ax=ax, cmap=cmap, vmin=-0.5, vmax=2.5, add_colorbar=False)
ax.set_title("Coherence-based damage severity — Amatrice, 24 Aug 2016")
ax.set_xlabel("")
ax.set_ylabel("")

patches = [mpatches.Patch(color=colors[i], label=labels[i]) for i in range(3)]
fig.legend(handles=patches, bbox_to_anchor=(0.95, 0.3), loc=1)
plt.tight_layout()
plt.show()


## Notes & limitations

- This is a **rapid-response damage proxy**, not a validated damage-grade classification —
  the class thresholds (`DELTA_COH_PARTIAL`, `DELTA_COH_SEVERE`, `DELTA_AMP_THRESHOLD`) should
  be calibrated against known-damaged reference areas before operational use, and will need
  retuning per terrain type (dense urban vs. rural stone-built villages, as around Amatrice).
- The **baseline pair** matters: choosing two pre-event dates that are themselves unusually
  decorrelated (e.g. due to rain, snow, or agricultural activity) will bias `delta_coherence`
  — picking a stable, dry, vegetation-quiet baseline period improves reliability.
- Non-seismic sources of coherence loss (vegetation growth, freshly ploughed fields, snow)
  are not excluded here; combining with a land-cover mask is a cheap way to reduce false
  positives without adding any InSAR time-series inversion.
- Everything in this workflow is pairwise (two independent coherence pairs + two amplitude
  dates) — still no phase unwrapping, no displacement, no SBAS/PSI network inversion.
